# Experiment Fixes

This notebook applies three fixes identified from reviewing Experiment A and B results:

**Fix 1 — yes_rate diagnostic:** Load `experiment_B_results.json` and check whether SmolLM3 and Qwen2.5 have a degenerate yes-bias (yes_rate > 0.85). If so, reframe them as a capability-threshold finding rather than invalid results.

**Fix 2 — Model table reframing:** Add a yes_rate column to the published table and add a note that sub-3.8B models collapse to uniform output bias on balanced binary tasks.

**Fix 3 — Reflexivity prompt fix:** Rerun ONLY the reflexivity evaluation from Experiment A with a corrected prompt that adds an explicit self-referential premise. This makes reflexivity a logical test rather than a factual-knowledge query.

In [ ]:
# Run this from the SP-25 directory (same as other notebooks)
import os
if os.path.exists('SP-25'):
    %cd SP-25
elif not os.path.exists('datasets'):
    !git clone https://github.com/Atharvy700/SP-25.git
    %cd SP-25

!pip install transformers accelerate -q

import json, random, gc, torch
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
from transformers import AutoTokenizer, AutoModelForCausalLM

random.seed(42)
np.random.seed(42)
print('Setup complete.')

---
## Fix 1: yes_rate Diagnostic

A model that always outputs "yes" will score 100% on reflexivity and symmetry (where all gold labels are yes) and exactly 50% on transitivity (50/50 split). The yes_rate field in experiment_B_results.json exposes this.

In [ ]:
# ── Upload experiment_B_results.json ──────────────────────────────────────────
from google.colab import files

print('Select experiment_B_results.json from your computer...')
uploaded = files.upload()

if 'experiment_B_results.json' in uploaded:
    print('Upload successful.')
else:
    print('WARNING: experiment_B_results.json not found in upload.')
    print('Make sure you selected the correct file.')

In [ ]:
# ── Load Experiment B results and check yes_rate ──────────────────────────────
RESULTS_PATH = 'experiment_B_results.json'

if not os.path.exists(RESULTS_PATH):
    print(f'ERROR: {RESULTS_PATH} not found.')
    print('Make sure experiment_B_domain_breakdown.ipynb has been run and')
    print('experiment_B_results.json has been downloaded to this directory.')
else:
    with open(RESULTS_PATH) as f:
        exp_b = json.load(f)

    models  = exp_b['models']
    domains = exp_b['domains']
    results = exp_b['results']

    YES_BIAS_THRESHOLD = 0.85

    print('=' * 70)
    print('YES_RATE DIAGNOSTIC — per model, per domain, per property')
    print(f'Threshold for yes-bias: > {YES_BIAS_THRESHOLD}')
    print('=' * 70)

    biased_models = []

    for mlabel in models:
        print(f'\n[{mlabel}]')
        model_yes_rates = []
        for dk in domains:
            for prop in ['reflexivity', 'symmetry', 'transitivity']:
                cell = results[mlabel][dk][prop]
                yr = cell.get('yes_rate', None)
                if yr is not None:
                    model_yes_rates.append(yr)
                    flag = ' <<< YES BIAS' if yr > YES_BIAS_THRESHOLD else ''
                    print(f'  {dk:<20} {prop:<14} yes_rate={yr:.3f}{flag}')

        if model_yes_rates:
            avg_yr = np.mean(model_yes_rates)
            print(f'  Average yes_rate: {avg_yr:.3f}')
            if avg_yr > YES_BIAS_THRESHOLD:
                biased_models.append(mlabel)
                print(f'  VERDICT: YES-BIASED — results are not valid for balanced evaluation')
            else:
                print(f'  VERDICT: OK — output distribution is not degenerate')

    print()
    print('=' * 70)
    if biased_models:
        print(f'BIASED MODELS: {biased_models}')
        print('Action: Reframe as capability-threshold finding (see Fix 2).')
    else:
        print('No degenerate yes-bias detected. All model results are valid.')
    print('=' * 70)

In [ ]:
# ── Visualize yes_rate per model ──────────────────────────────────────────────
if os.path.exists(RESULTS_PATH):
    props   = ['reflexivity', 'symmetry', 'transitivity']
    n_props = len(props)
    colors  = ['#2196F3', '#FF5722', '#4CAF50']

    fig, axes = plt.subplots(1, n_props, figsize=(16, 5))
    fig.suptitle(
        'Yes-Output Rate by Model, Domain, and Property\n'
        'Dashed line = 0.85 threshold. Models above this across all domains have degenerate yes-bias.',
        fontsize=12, fontweight='bold'
    )

    x = np.arange(len(domains))
    width = 0.8 / len(models)

    for ax, prop in zip(axes, props):
        for i, (mlabel, color) in enumerate(zip(models, colors)):
            yes_rates = []
            for dk in domains:
                cell = results[mlabel][dk][prop]
                yes_rates.append(cell.get('yes_rate', 0.0))
            offset = (i - len(models)/2 + 0.5) * width
            ax.bar(x + offset, yes_rates, width, label=mlabel, color=color, alpha=0.85)

        ax.axhline(YES_BIAS_THRESHOLD, color='red', linestyle='--', linewidth=1.5,
                   alpha=0.8, label=f'Bias threshold ({YES_BIAS_THRESHOLD})')
        ax.axhline(0.5, color='gray', linestyle=':', linewidth=1.2,
                   alpha=0.6, label='Balanced (0.5)')
        ax.set_xticks(x)
        ax.set_xticklabels([d.replace('_', '\n') for d in domains], fontsize=8)
        ax.set_ylim(0, 1.1)
        ax.set_ylabel('Yes-output rate', fontsize=10)
        ax.set_title(prop.capitalize(), fontsize=11, fontweight='bold')
        ax.legend(fontsize=7)
        ax.grid(axis='y', alpha=0.3)

    plt.tight_layout()
    plt.savefig('fix1_yesrate_diagnostic.png', dpi=150, bbox_inches='tight')
    plt.show()
    print('Saved: fix1_yesrate_diagnostic.png')

---
## Fix 2: Model Table Reframing

If SmolLM3 and Qwen2.5 are yes-biased, we do not discard their results. Instead we:
1. Add a yes_rate column to the published table — this is transparent and reviewers will appreciate it
2. Reframe the finding: models below a capability threshold collapse to uniform output bias on balanced binary tasks, rendering per-question accuracy uninformative
3. This becomes a finding about **capability thresholds** for relational reasoning benchmarks

In [ ]:
# ── Print reframed Table 1 with yes_rate column ───────────────────────────────
if os.path.exists(RESULTS_PATH):
    props = ['reflexivity', 'symmetry', 'transitivity']

    print('TABLE 1 (REVISED): Accuracy (%) and Yes-Output Rate by Model and Property')
    print('Accuracy averaged across all 6 semantic domains.')
    print('Yes-rate > 0.85 indicates degenerate output bias (marked with *).')
    print()
    print(f'{"Model":<25} {"Refl Acc":>9} {"Sym Acc":>9} {"Trans Acc":>10} '
          f'{"Refl Yes%":>10} {"Sym Yes%":>9} {"Trans Yes%":>11}')
    print('-' * 90)

    for mlabel in models:
        row_accs = []
        row_yrs  = []
        for prop in props:
            accs = [results[mlabel][dk][prop]['acc'] for dk in domains]
            yrs  = [results[mlabel][dk][prop].get('yes_rate', 0) for dk in domains]
            row_accs.append(np.mean(accs) * 100)
            row_yrs.append(np.mean(yrs))

        bias_flag = '*' if any(yr > YES_BIAS_THRESHOLD for yr in row_yrs) else ' '
        print(f'{bias_flag}{mlabel:<24}', end='')
        for acc in row_accs:
            print(f' {acc:>9.1f}', end='')
        for yr in row_yrs:
            flag = '†' if yr > YES_BIAS_THRESHOLD else ' '
            print(f' {yr*100:>9.1f}{flag}', end='')
        print()

    print()
    print('* Model shows degenerate yes-bias (avg yes_rate > 85%).')
    print('† Accuracy inflated by yes-bias; not a valid measure of reasoning ability.')
    print()
    print('PAPER NOTE: Add this footnote to Table 1:')
    print('"Models marked * output yes-bias rates above 85%, indicating collapse to a')
    print('uniform output distribution rather than genuine binary classification.')
    print('Their symmetry and reflexivity accuracies are inflated by label imbalance')
    print('in those sub-tasks (all positive examples) and should not be interpreted as')
    print('evidence of reasoning competence. This identifies a capability threshold:')
    print('models below ~3.8B parameters cannot reliably perform calibrated binary')
    print('inference on this benchmark without explicit output normalization."')

---
## Fix 3: Reflexivity Prompt Fix + Rerun

**The problem:** The current reflexivity prompt for semantic domains is:
```
Is KFQZ the same age as KFQZ?
```
This is a factual question the model cannot answer (it has no knowledge of KFQZ's age), so it defaults to "no." The 0% accuracy on kinship/spatial reflexivity is a factual-grounding failure, not a logical failure.

**The fix:** Add a self-referential premise that makes the logical structure explicit:
```
Suppose KFQZ is the same age as KFQZ. Is KFQZ the same age as KFQZ?
```
This forces the model to answer from the given premise, not from world knowledge. A logically consistent model must answer "yes."

**Only reflexivity is rerun** — symmetry and transitivity prompts are not affected.

In [ ]:
# ── Fixed domain templates (reflexivity prompts only) ─────────────────────────
# The key change: each reflexivity question now includes an explicit premise
# "Suppose X R X." before asking "Is X R X?"
# This removes the factual-knowledge dependency and makes it purely logical.

DOMAINS_FIXED = {
    'abstract': {
        'refl_premise': 'Suppose {A} equals {A}.',
        'refl_q':       'Does {A} equal {A}?',
        'label': 'Abstract',
    },
    'kinship': {
        'refl_premise': 'Suppose {A} is the same age as {A}.',
        'refl_q':       'Is {A} the same age as {A}?',
        'label': 'Kinship',
    },
    'set_theory': {
        'refl_premise': 'Suppose set {A} has the same cardinality as set {A}.',
        'refl_q':       'Does set {A} have the same cardinality as set {A}?',
        'label': 'Set Theory',
    },
    'object_identity': {
        'refl_premise': 'Suppose object {A} is identical to object {A}.',
        'refl_q':       'Is object {A} identical to object {A}?',
        'label': 'Object Identity',
    },
    'synonymy': {
        'refl_premise': 'Suppose the term {A} is synonymous with the term {A}.',
        'refl_q':       'Is the term {A} synonymous with the term {A}?',
        'label': 'Synonymy',
    },
    'spatial': {
        'refl_premise': 'Suppose location {A} is at the same coordinates as location {A}.',
        'refl_q':       'Is location {A} at the same coordinates as location {A}?',
        'label': 'Spatial',
    },
}

INSTRUCTION = 'You are a logical reasoning assistant.\nAnswer with exactly one word.\n\n'
SUFFIX      = '\n\nAnswer only yes or no:'

# Show the difference between old and new prompts
print('PROMPT COMPARISON — Kinship domain')
print()
print('OLD (factual, fails):')
print(INSTRUCTION + 'Is KFQZ the same age as KFQZ?' + SUFFIX)
print()
print('NEW (logical, fixed):')
print(INSTRUCTION + 'Suppose KFQZ is the same age as KFQZ. Is KFQZ the same age as KFQZ?' + SUFFIX)

In [ ]:
# ── Entity name pool ──────────────────────────────────────────────────────────
def gen_names(n=50000, lo=4, hi=7, seed=42):
    rng = random.Random(seed)
    pool = set()
    while len(pool) < n:
        k = rng.randint(lo, hi)
        pool.add(''.join(rng.choices('ABCDEFGHIJKLMNOPQRSTUVWXYZ', k=k)))
    return list(pool)

NAMES = gen_names()
pool  = NAMES[:8000]
rng   = random.Random(42)
print(f'Name pool ready: {len(pool)} entities.')

In [ ]:
# ── Load Phi-3-mini (only model needed for reflexivity rerun) ─────────────────
if not torch.cuda.is_available():
    raise RuntimeError('Enable GPU: Runtime → Change runtime type → GPU')

gc.collect()
torch.cuda.empty_cache()

MODEL_NAME = 'microsoft/Phi-3-mini-4k-instruct'
MAX_LENGTH  = 256   # reflexivity prompts are short

print('Loading tokenizer...')
tok = AutoTokenizer.from_pretrained(MODEL_NAME)
if tok.pad_token is None:
    tok.pad_token = tok.eos_token

print('Loading model (FP16)...')
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME, torch_dtype=torch.float16, device_map='auto'
)
model.eval()

YES_ID = tok('yes', add_special_tokens=False).input_ids[0]
NO_ID  = tok('no',  add_special_tokens=False).input_ids[0]
print(f'yes_id={YES_ID}  no_id={NO_ID}\nModel ready.')

In [ ]:
@torch.inference_mode()
def predict_batch(prompts, batch_size=16):
    results = []
    for i in range(0, len(prompts), batch_size):
        batch = prompts[i:i+batch_size]
        enc = tok(
            batch, return_tensors='pt', padding=True,
            truncation=True, max_length=MAX_LENGTH
        ).to(model.device)
        logits = model(**enc).logits[:, -1, :]
        for lg in logits:
            results.append('yes' if lg[YES_ID] > lg[NO_ID] else 'no')
    return results

In [ ]:
# ── Rerun reflexivity with OLD vs NEW prompts side by side ────────────────────
N_REFL = 300
entities = rng.sample(pool, N_REFL)

old_results = {}
new_results = {}

for dk, dv in DOMAINS_FIXED.items():
    print(f'Domain: {dv["label"]}')

    # OLD prompt (no premise)
    old_prompts = [
        INSTRUCTION + dv['refl_q'].format(A=X) + SUFFIX
        for X in entities
    ]
    old_preds = predict_batch(old_prompts)
    old_acc   = np.mean([p == 'yes' for p in old_preds])
    old_results[dk] = old_acc

    # NEW prompt (with explicit self-referential premise)
    new_prompts = [
        INSTRUCTION + dv['refl_premise'].format(A=X) + ' ' + dv['refl_q'].format(A=X) + SUFFIX
        for X in entities
    ]
    new_preds = predict_batch(new_prompts)
    new_acc   = np.mean([p == 'yes' for p in new_preds])
    new_results[dk] = new_acc

    improvement = new_acc - old_acc
    print(f'  Old accuracy: {old_acc:.4f}')
    print(f'  New accuracy: {new_acc:.4f}')
    print(f'  Improvement:  {improvement:+.4f}\n')

print('Reflexivity rerun complete.')

In [ ]:
# ── Plot: old vs new reflexivity accuracy ─────────────────────────────────────
domain_keys    = list(DOMAINS_FIXED.keys())
domain_labels  = [DOMAINS_FIXED[dk]['label'] for dk in domain_keys]
old_accs = [old_results[dk] for dk in domain_keys]
new_accs = [new_results[dk] for dk in domain_keys]

x     = np.arange(len(domain_keys))
width = 0.35

fig, ax = plt.subplots(figsize=(12, 6))
bars1 = ax.bar(x - width/2, old_accs, width,
               label='Old prompt (no premise)', color='#F44336', alpha=0.85)
bars2 = ax.bar(x + width/2, new_accs, width,
               label='New prompt (self-referential premise)', color='#4CAF50', alpha=0.85)

ax.axhline(1.0, color='green', linestyle='--', alpha=0.5, linewidth=1.5, label='Perfect (1.0)')
ax.axhline(0.5, color='gray',  linestyle=':', alpha=0.6, linewidth=1.2, label='Chance (0.5)')

for bar, val in zip(bars1, old_accs):
    ax.text(bar.get_x() + bar.get_width()/2, val + 0.02,
            f'{val:.3f}', ha='center', fontsize=9, color='#F44336', fontweight='bold')
for bar, val in zip(bars2, new_accs):
    ax.text(bar.get_x() + bar.get_width()/2, val + 0.02,
            f'{val:.3f}', ha='center', fontsize=9, color='#2E7D32', fontweight='bold')

ax.set_xticks(x)
ax.set_xticklabels(domain_labels, fontsize=11)
ax.set_ylim(0, 1.15)
ax.set_ylabel('Accuracy (fraction answered yes)', fontsize=11)
ax.set_title(
    'Reflexivity Accuracy: Old Prompt vs. Fixed Prompt\n'
    'Phi-3-mini-4k-instruct — 300 randomly-named entities per domain\n'
    'Old: "Is X R X?"  |  New: "Suppose X R X. Is X R X?"',
    fontsize=12, fontweight='bold'
)
ax.legend(fontsize=10)
ax.grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.savefig('fix3_reflexivity_comparison.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: fix3_reflexivity_comparison.png')

In [ ]:
# ── Interpretation ────────────────────────────────────────────────────────────
print('REFLEXIVITY RESULTS INTERPRETATION')
print('=' * 60)
print()
print(f'{"Domain":<20} {"Old Acc":>9} {"New Acc":>9} {"Delta":>8}')
print('-' * 50)
for dk in domain_keys:
    label = DOMAINS_FIXED[dk]['label']
    old   = old_results[dk]
    new   = new_results[dk]
    delta = new - old
    print(f'{label:<20} {old:>9.4f} {new:>9.4f} {delta:>+8.4f}')
print()

# Classify each domain
for dk in domain_keys:
    label = DOMAINS_FIXED[dk]['label']
    new   = new_results[dk]
    old   = old_results[dk]
    if new > 0.90:
        finding = 'PASSES reflexivity when premise is given'
    elif new > 0.60:
        finding = 'PARTIAL — still some failures even with premise'
    else:
        finding = 'FAILS reflexivity even with explicit premise — deeper issue'
    print(f'{label}: {finding}')

print()
print('PAPER INTERPRETATION:')
print('If new_acc is high (>0.9): The old 0% result was a prompt design artifact.')
print('  The model CAN apply reflexivity when asked logically rather than factually.')
print('  Update the paper to use the new prompt and replace the reflexivity table.')
print()
print('If new_acc is still low (<0.5): The failure is genuine — the model cannot')
print('  apply the reflexivity axiom even when the premise is stated explicitly.')
print('  This is a stronger finding. Report both old and new numbers.')

In [ ]:
# ── Negative control: add premise for two DIFFERENT entities ──────────────────
# If the model says yes to "Suppose X R X. Is X R X?" for all domains,
# we need to verify it is not just saying yes to any premised question.
# Control: "Suppose X R Y (X != Y). Is X R X?" — should still be NO.

print('NEGATIVE CONTROL: Does adding a premise make model say yes to everything?')
print('Control prompt: "Suppose A equals B (A!=B). Does A equal A?" — correct answer: YES')
print('(Reflexivity must hold regardless of what else is true)')
print()

entities_A = rng.sample(pool, 100)
entities_B = rng.sample(pool, 100)
# Ensure A != B
pairs = [(a, b) for a, b in zip(entities_A, entities_B) if a != b][:100]

control_prompts = [
    INSTRUCTION +
    f'Suppose {A} equals {B}. Does {A} equal {A}?' +
    SUFFIX
    for A, B in pairs
]
control_preds = predict_batch(control_prompts)
control_acc   = np.mean([p == 'yes' for p in control_preds])

print(f'Control accuracy (abstract domain): {control_acc:.4f}')
if control_acc > 0.8:
    print('GOOD: Model correctly answers yes to reflexivity even with an unrelated premise.')
else:
    print('NOTE: Model does not reliably answer yes to reflexivity in this control.')
    print('This suggests the model is not applying logical axioms consistently.')

In [ ]:
# ── Save all fix results ──────────────────────────────────────────────────────
fix_results = {
    'fix1_yes_rate_threshold': YES_BIAS_THRESHOLD,
    'fix1_biased_models': biased_models if os.path.exists(RESULTS_PATH) else 'experiment_B_results.json not found',
    'fix3_reflexivity': {
        dk: {
            'old_acc': round(old_results[dk], 4),
            'new_acc': round(new_results[dk], 4),
            'delta':   round(new_results[dk] - old_results[dk], 4),
        }
        for dk in domain_keys
    },
    'fix3_negative_control_acc': round(float(control_acc), 4),
}

with open('experiment_fixes_results.json', 'w') as f:
    json.dump(fix_results, f, indent=2)

print('Saved: experiment_fixes_results.json')
print()
print(json.dumps(fix_results, indent=2))

---
## What to update in the paper based on these results

### If yes_rate > 0.85 for SmolLM3 and Qwen2.5 (Fix 1 + Fix 2)

Change the Model section of the paper to read:

> We evaluate three open-weight models: Phi-3-mini-4k-instruct (3.8B), SmolLM3-3B (3B), and Qwen2.5-0.5B-Instruct (0.5B). Table 1 reports accuracy alongside yes-output rate per property. SmolLM3 and Qwen2.5 exhibit yes-output rates above 85% across all conditions, indicating collapse to a uniform output distribution rather than calibrated binary inference. Their reflexivity and symmetry accuracies are inflated by label imbalance in those sub-tasks and should not be interpreted as evidence of reasoning ability. We report their results for completeness and as a finding about capability thresholds: models below approximately 3.8B parameters fail to produce calibrated binary outputs on this benchmark under zero-shot prompting.

### If new reflexivity accuracy is high (Fix 3)

> The original reflexivity evaluation presented questions of the form "Is X the same age as X?" without an explicit premise. Under this formulation, the model may interpret the question as a factual inquiry rather than a logical one, defaulting to "no" because it has no world-knowledge about the entity. We corrected the prompt to include an explicit self-referential premise ("Suppose X is the same age as X. Is X the same age as X?"), which makes the logical structure unambiguous. Under the corrected prompt, reflexivity accuracy rises substantially across semantic domains. We report both results in Table 2 and use the corrected prompt for all final evaluations.

### If new reflexivity accuracy is still low (Fix 3)

> Even with an explicit self-referential premise, the model fails to consistently apply the reflexivity axiom in several domains. This indicates a genuine failure to treat logical axioms as constraints rather than factual claims — a deeper and more significant finding than the prompt design issue alone.